# ADRec: Diffusion Steps, Accuracy, and Inference Time

Train 8- and 16-step ADRec variants for seeds 42–46. Each variant starts from that seed's existing pretrained item embeddings. Load the existing 32-step ADRec checkpoint as the baseline; do not retrain it or pretraining.

Attach three private inputs to Kaggle:
1. The current `cmpe257_kaggle_code.zip` (or its extracted contents).
2. The selected dataset’s `dataset.pkl` and its matching five-seed baseline checkpoints.
3. `adrec_all_results_and_models.zip`, downloaded from the completed five-seed run. It contains the result ZIP and five checkpoint ZIPs.

Import this notebook, enable Internet for dependency installation, and select a GPU. Run the numbered cells in order. Inputs remain unchanged; experiment outputs go into a separate directory.

This is a sensitivity study of the authors' **truncated-linear schedule at 8, 16, and 32 steps**. The schedule changes its endpoint formula at small step counts, so differences cannot be attributed solely to iteration count. Accuracy improvement is not assumed. Keep the original five-seed results as the reproduction baseline.
Select `DATASET` in section 1. Beauty and MovieLens require their own completed 32-step ADRec and pretraining runs before this optional experiment. Set `CHECKPOINT_INPUT` to that dataset’s downloaded exports. Baby experiments do not need to be rerun. New dataset outputs and ZIPs use separate dataset folders.


## 1. Load the code and dataset
Set explicit paths when multiple versions of an input are attached. Start a fresh kernel when changing source bundles.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, sys, zipfile

# None enables automatic discovery within the attached Kaggle inputs.
# Set explicit paths when multiple matching files are attached.
CODE_ZIP = None  # Path to the attached code ZIP
CODE_DIR = None  # Optional extracted folder containing bundle_manifest.json
DATASET = 'baby'  # 'baby', 'beauty', or 'ml-100k'
DATA_FILE = None  # Optional path to the selected dataset.pkl
INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')

def attached_file(name, override):
    if override:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    matches = sorted(INPUT_ROOT.rglob(name))
    if len(matches) != 1:
        raise ValueError(f'Expected one {name}; found {matches}. Set its path above.')
    return matches[0]

REPO = WORK_ROOT / 'CMPE257_ML'
REPO.mkdir(parents=True, exist_ok=True)
zip_matches = sorted(INPUT_ROOT.rglob('cmpe257_kaggle_code.zip'))
archive = None
if CODE_DIR is None and (CODE_ZIP is not None or zip_matches):
    archive = attached_file('cmpe257_kaggle_code.zip', CODE_ZIP)
    with zipfile.ZipFile(archive) as z:
        for member in z.namelist():
            if not (REPO / member).resolve().is_relative_to(REPO.resolve()):
                raise ValueError('Unsafe archive member')
        z.extractall(REPO)
else:
    # Kaggle may unpack the uploaded ZIP before attaching it to a notebook.
    manifest_path = (Path(CODE_DIR) / 'bundle_manifest.json' if CODE_DIR
                     else attached_file('bundle_manifest.json', None))
    source_dir = manifest_path.parent.resolve()
    manifest = json.loads(manifest_path.read_text())
    for name in [*manifest['files'], 'bundle_manifest.json']:
        source, target = (source_dir / name).resolve(), (REPO / name).resolve()
        if not source.is_relative_to(source_dir) or not target.is_relative_to(REPO.resolve()):
            raise ValueError('Unsafe manifest path')
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(source, target)
manifest = json.loads((REPO / 'bundle_manifest.json').read_text())
for name, expected in manifest['files'].items():
    file = (REPO / name).resolve()
    if not file.is_relative_to(REPO.resolve()):
        raise ValueError('Unsafe manifest path')
    assert hashlib.sha256(file.read_bytes()).hexdigest() == expected, name
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('Source branch:', manifest['branch'])
print('Base commit:', manifest['base_commit'])
if archive:
    print('Source ZIP SHA256:', hashlib.sha256(archive.read_bytes()).hexdigest())
else:
    print('Loaded Kaggle-extracted source; all manifest hashes verified.')
from src.data.catalog import find_dataset, dataset_spec
DATASET_PATH = find_dataset(INPUT_ROOT, DATASET, DATA_FILE)
ARCHIVE_ROOT = WORK_ROOT if DATASET == 'baby' else WORK_ROOT / 'exports' / DATASET
ARCHIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Dataset:', DATASET, DATASET_PATH)

## 2. Install dependencies
Install the shared dependencies before importing the models. Restart the kernel only if installation requires it, then rerun setup.

In [ ]:
import subprocess
# Keep Kaggle's CUDA-enabled torch if it satisfies the shared requirements.
# Internet must be enabled for missing dependencies. Run before importing models.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '--upgrade-strategy', 'only-if-needed',
                '-r', str(REPO / 'requirements-shared.txt')], check=True)

## 3. Load the runtime

In [ ]:
import torch
from src.data import load_dataset, SequenceDataset
from src.data.load import dataset_summary
from src.training.kaggle import make_config, train, load_for_inference, predict_scores
from src.utils.reproducibility import seed_everything

print('PyTorch:', torch.__version__, 'CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    torch.set_num_threads(1)
    print('CPU is sufficient for smoke mode. Enable a GPU before full training.')
# Check model imports before starting training.
from sasrec import SASRec
from model import Att_Diffuse_model

from src.evaluation import evaluate

## 4. Configure the experiment

Full mode trains ten variants: two step counts × five seeds. `RUN_MODE = 'smoke'` runs two epochs on nine users with the existing full-sized embeddings; it verifies the pipeline and does not produce comparable accuracy results. Smoke mode uses the same step counts but shorter training and different evaluation batching.

`CHECKPOINT_INPUT` accepts the combined download ZIP or a directory containing the five `adrec_full_seed<N>.zip` files and `adrec_results.zip`. Extracted directories named `adrec_full_seed<N>/` and `adrec_results/` are also accepted. `None` discovers the combined ZIP or seed-42 archive/folder below `/kaggle/input`.

`RESUME_ROOT` optionally points to an extracted previous experiment directory containing `runs/`. Completed runs are reused, interrupted runs resume from their latest saved epoch, and configurations are checked. Checkpoints from the original 32-step experiment are never overwritten.

In [ ]:
RUN_MODE = 'full'
SEEDS = [42, 43, 44, 45, 46]
TRAIN_STEPS = [8, 16]
CHECKPOINT_INPUT = None
RESUME_ROOT = None

BENCHMARK_BATCH_SIZES = [1, 32]
BENCHMARK_USERS = 128
WARMUP_REPEATS = 5
TIMED_REPEATS = 20
BENCHMARK_SEED = 2025

assert RUN_MODE in ('full', 'smoke')
assert SEEDS and len(SEEDS) == len(set(SEEDS))
assert all(type(s) is int and s in range(42, 47) for s in SEEDS)
assert TRAIN_STEPS == [8, 16]
if RUN_MODE == 'full' and not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU for full training.')
if RUN_MODE == 'smoke':
    BENCHMARK_BATCH_SIZES = [1, 4]
    BENCHMARK_USERS, WARMUP_REPEATS, TIMED_REPEATS = 8, 1, 2
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
EXPERIMENT_ROOT = WORK_ROOT / f'adrec_diffusion_steps_{RUN_MODE}'
if DATASET != 'baby':
    EXPERIMENT_ROOT /= DATASET
RUNS_ROOT = EXPERIMENT_ROOT / 'runs'
RESULTS_DIR = EXPERIMENT_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
reference_cfg = make_config('adrec', 'full', dataset=DATASET)
data = load_dataset(DATASET_PATH, reference_cfg['dataset_sha256'])
print('Mode:', RUN_MODE, '| Seeds:', SEEDS, '| New step counts:', TRAIN_STEPS)
print('Output:', EXPERIMENT_ROOT)

## 5. Define input validation and measurement functions

In [ ]:
import csv, gc, io, random, statistics, time
from datetime import datetime, timezone
import numpy as np

METRICS = ['HR@5', 'HR@10', 'HR@20', 'NDCG@5', 'NDCG@10', 'NDCG@20']

def read_json(path):
    return json.loads(Path(path).read_text())

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2) + '\n')
    temporary.replace(path)

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def unpack(source, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(source) as archive:
        for info in archive.infolist():
            target = (destination / info.filename).resolve()
            if not target.is_relative_to(destination) or (info.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError(f'Unsafe ZIP entry: {info.filename}')
        archive.extractall(destination)
    return destination

def unique(paths, description):
    paths = sorted(set(Path(p).resolve() for p in paths))
    if len(paths) != 1:
        raise ValueError(f'Expected one {description}; found {paths}. Set CHECKPOINT_INPUT.')
    return paths[0]

def locate_references(override, input_root, cache, seeds, dataset="baby"):
    if dataset != "baby" and override is None:
        raise ValueError("Set CHECKPOINT_INPUT to the selected dataset baseline exports")
    if override is None:
        combined = list(Path(input_root).rglob('adrec_all_results_and_models.zip'))
        if combined:
            source = unique(combined, 'combined checkpoint ZIP')
        else:
            candidates = [p.parent for p in Path(input_root).rglob('adrec_full_seed42.zip')]
            candidates += [p.parent for p in Path(input_root).rglob('adrec_full_seed42') if p.is_dir()]
            source = unique(candidates, 'checkpoint input directory')
    else:
        source = Path(override).resolve()
    if source.is_file():
        source = unpack(source, Path(cache) / sha256(source)[:16])
    if not source.is_dir():
        raise FileNotFoundError(source)
    result_sources = [source / 'adrec_results', source / 'results/adrec']
    metadata = next((p for p in result_sources if p.is_dir()), None)
    if metadata is None:
        archive = source / 'adrec_results.zip'
        metadata = unpack(archive, Path(cache) / ('results_' + sha256(archive)[:16]))
    references = {}
    for seed in seeds:
        name = f'adrec_full_seed{seed}'
        folder = source / name
        if not folder.is_dir():
            archive = source / (name + '.zip')
            folder = unpack(archive, Path(cache) / (name + '_' + sha256(archive)[:16]))
        references[seed] = {'run': folder, 'metadata': metadata}
    return references

def check_reference(seed, ref, expected_cfg, repo):
    run, metadata = ref['run'], ref['metadata']
    cfg = read_json(run / 'adrec/config.json')
    if cfg != dict(expected_cfg, seed=seed):
        raise ValueError(f'Seed {seed}: baseline settings differ from the current full configuration')
    pre_cfg = read_json(run / 'pretrain/config.json')
    if pre_cfg != dict(expected_cfg, model='pretrain', seed=seed):
        raise ValueError(f'Seed {seed}: pretraining settings mismatch')
    record = read_json(metadata / f'adrec_seed{seed}_checkpoints.json')
    for stage in ('pretrain', 'adrec'):
        key = f'{stage}/best.pt'
        if sha256(run / key) != record['files'][key]:
            raise ValueError(f'Seed {seed}: checkpoint checksum mismatch for {key}')
        saved = torch.load(run / key, map_location='cpu', weights_only=True)
        expected = pre_cfg if stage == 'pretrain' else cfg
        if saved['config'] != expected or saved['num_items'] != dataset_summary(data)['max_item_id']:
            raise ValueError(f'Seed {seed}: checkpoint configuration mismatch')
        if tuple(saved['model_state']['item_embedding.weight'].shape) != (saved['num_items'] + 1, cfg['hidden_size']):
            raise ValueError('Embedding dimensions do not match the dataset')
    report = read_json(run / 'adrec/results.json')
    if report != read_json(metadata / f'adrec_seed{seed}_results.json'):
        raise ValueError('Baseline result exports disagree')
    if report['test']['protocol'] != 'adrec_upstream' or report['test']['units'] != 'percent':
        raise ValueError('Baseline evaluation protocol mismatch')
    env = read_json(run / 'adrec/environment.json')
    if env['pretrained_checkpoint']['sha256'] != sha256(run / 'pretrain/best.pt'):
        raise ValueError('Baseline was not trained from the supplied pretrained checkpoint')
    migrations = check_source_compatibility(env['source']['files'], repo, cfg.get('dataset', 'baby'))
    ref['source_migrations'] = migrations
    return cfg


def check_source_compatibility(files, repo, dataset):
    # Only exact audited old/new hash pairs are accepted for legacy Baby runs.
    registry = read_json(Path(repo) / 'configs/legacy_source_compatibility.json')
    accepted = []
    for name, digest in files.items():
        if name.startswith(('src/', 'configs/', 'third_party/')) and name.endswith(('.py', '.json', '.yaml')):
            current = sha256(Path(repo) / name)
            if current == digest:
                continue
            migration = next((m for m in registry['migrations']
                              if dataset == m['dataset'] and name == m['path']
                              and digest == m['before'] and current == m['after']), None)
            if migration is None:
                raise ValueError(f'Training/evaluation source differs from baseline: {name}')
            accepted.append(migration)
    return accepted

def variant_config(base, steps, mode):
    cfg = dict(base, diffusion_steps=steps)
    if mode == 'smoke':
        cfg.update(mode='smoke', user_limit=9, epochs=2, train_batch_size=4,
                   evaluation_batch_size=4, eval_interval=1, warmup_epochs=1)
    return cfg

def ensure_variant(cfg, pretrain, output, prior=None):
    output = Path(output)
    provenance = {'pretrained_sha256': sha256(pretrain), 'config': cfg}
    if not (output / 'latest.pt').exists() and prior and Path(prior).is_dir():
        if output.exists() and any(output.iterdir()):
            raise ValueError(f'Cannot copy a resume run into a nonempty directory: {output}')
        shutil.copytree(prior, output, dirs_exist_ok=True)
    output.mkdir(parents=True, exist_ok=True)
    marker = output / 'experiment_input.json'
    if marker.exists() and read_json(marker) != provenance:
        raise ValueError(f'Run inputs changed: {output}')
    if (output / 'latest.pt').exists() and not marker.exists():
        raise ValueError(f'Resume run is missing provenance: {output}')
    write_json(marker, provenance)
    resume = output / 'latest.pt'
    if resume.exists():
        if read_json(output / 'config.json') != cfg:
            raise ValueError(f'Resume configuration mismatch: {output}')
        report_path = output / 'results.json'
        report = read_json(report_path) if report_path.exists() else None
        complete = report and (report['early_stopped'] or report['epochs_completed'] >= cfg['epochs'])
        if complete and report.get('test') and (output / 'best.pt').exists():
            print('Reusing completed:', output.name)
            return report
    return train(cfg, DATASET_PATH, output, pretrained_checkpoint=pretrain,
                 resume=resume if resume.exists() else None, device=DEVICE, finalize=True)

def save_csv(path, rows):
    if not rows:
        raise ValueError('No rows to export')
    with Path(path).open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]), lineterminator='\n')
        writer.writeheader()
        writer.writerows(rows)

def summarize_rows(rows, group_keys, value_keys):
    groups = {}
    for row in rows:
        groups.setdefault(tuple(row[k] for k in group_keys), []).append(row)
    summaries = []
    for key, group in sorted(groups.items()):
        seeds = [r['seed'] for r in group]
        if len(seeds) != len(set(seeds)):
            raise ValueError('Duplicate seed within comparison group')
        summary = dict(zip(group_keys, key))
        summary.update(n_runs=len(group), seeds=','.join(map(str, sorted(seeds))))
        for metric in value_keys:
            values = [r[metric] for r in group]
            summary[metric + '_mean'] = statistics.mean(values)
            summary[metric + '_std'] = statistics.stdev(values) if len(values) > 1 else None
        summaries.append(summary)
    return summaries

def timing_batches(dataset, user_count, batch_size, device):
    if user_count < batch_size or batch_size < 1:
        raise ValueError('The timing user count must cover each batch size')
    indices = list(range(len(dataset)))
    random.Random(BENCHMARK_SEED).shuffle(indices)
    indices = indices[:min(user_count, len(indices))]
    batches = []
    for offset in range(0, len(indices) - batch_size + 1, batch_size):
        ids = indices[offset:offset + batch_size]
        batches.append(torch.tensor([dataset[i]['input_ids'] for i in ids], dtype=torch.long, device=device))
    if not batches:
        raise ValueError('Not enough users for a complete timing batch')
    return batches

def benchmark(model, batches, warmups, repeats, seed):
    if warmups < 1 or repeats < 2:
        raise ValueError('Use at least one warm-up and two measured repetitions')
    device = next(model.parameters()).device
    def synchronize():
        if device.type == 'cuda':
            torch.cuda.synchronize(device)
    def predict(x):
        scores = predict_scores(model, x)
        # Match benchmark candidate rules: no seen-item or padding filtering.
        return torch.topk(scores, 20, dim=-1).indices
    seed_everything(seed)
    model.eval()
    with torch.no_grad():
        for i in range(warmups):
            predict(batches[i % len(batches)])
        synchronize()
        samples = []
        for i in range(repeats):
            x = batches[i % len(batches)]
            synchronize()
            start = time.perf_counter()
            predict(x)
            synchronize()
            samples.append((time.perf_counter() - start) * 1000)
    mean_ms = statistics.mean(samples)
    return {'batch_size': len(batches[0]), 'mean_batch_ms': mean_ms,
            'median_batch_ms': statistics.median(samples),
            'p95_batch_ms': float(np.percentile(samples, 95)),
            'users_per_second': len(batches[0]) * 1000 / mean_ms,
            'measured_repeats': repeats, 'warmup_repeats': warmups,
            'timed_users': sum(len(x) for x in batches[:min(repeats, len(batches))]), 'samples_ms': samples}

## 6. Verify existing pretrained embeddings and 32-step checkpoints

Check the dataset, configurations, source hashes, checkpoint hashes, and the link between each baseline and its pretrained embeddings. A mismatch stops the experiment before training.

In [ ]:
references = locate_references(CHECKPOINT_INPUT, INPUT_ROOT, WORK_ROOT / 'adrec_reference_cache', SEEDS, dataset=DATASET)
baseline_configs = {}
for seed, ref in references.items():
    baseline_configs[seed] = check_reference(seed, ref, reference_cfg, REPO)
    print('Verified seed', seed, '| pretrain:', ref['run'] / 'pretrain/best.pt')

write_json(RESULTS_DIR / 'baseline_source_migrations.json',
           {str(seed): ref.get('source_migrations', []) for seed, ref in references.items()})

from src.training.kaggle import _args
from step_sample import get_named_beta_schedule
schedule_rows = []
for steps in [8, 16, 32]:
    cfg = dict(reference_cfg, diffusion_steps=steps)
    betas = np.asarray(get_named_beta_schedule(_args(cfg, dataset_summary(data)['max_item_id'], DEVICE)))
    schedule_rows.append({'diffusion_steps': steps, 'noise_schedule': 'trunc_lin',
                          'beta_start': float(betas[0]), 'beta_end': float(betas[-1]),
                          'terminal_alpha_bar': float(np.prod(1 - betas)),
                          'small_step_endpoint_adjustment': (1000 / steps * 0.02 + 0.01) > 1})
write_json(RESULTS_DIR / 'noise_schedules.json', schedule_rows)
print(json.dumps(schedule_rows, indent=2))

## 7. Train 8- and 16-step variants

Only the ADRec stage is trained. For a given seed, both variants load exactly the same pretrained embedding checkpoint as the existing 32-step model. Other full-mode settings stay fixed. Validation selects the checkpoint; the shared evaluator produces the final test metrics. The notebook does not automatically select a step count using test results.

Run folders and per-variant ZIPs have distinct names. Rerunning this cell reuses completed runs and resumes interrupted runs. A fresh Kaggle session needs the saved experiment folders through `RESUME_ROOT` to resume.

In [ ]:
variant_runs = {}
for seed in SEEDS:
    for steps in TRAIN_STEPS:
        cfg = variant_config(baseline_configs[seed], steps, RUN_MODE)
        name = f'adrec_steps{steps}_seed{seed}'
        run_dir = RUNS_ROOT / name
        prior = Path(RESUME_ROOT) / 'runs' / name if RESUME_ROOT else None
        print(f'Training ADRec: seed={seed}, steps={steps}, mode={RUN_MODE}', flush=True)
        ensure_variant(cfg, references[seed]['run'] / 'pretrain/best.pt', run_dir, prior)
        variant_runs[(seed, steps)] = run_dir
        export_dir = RESULTS_DIR / name
        export_dir.mkdir(exist_ok=True)
        for filename in ('config.json', 'history.json', 'results.json', 'environment.json',
                         'requirements-lock.txt', 'experiment_input.json'):
            shutil.copyfile(run_dir / filename, export_dir / filename)
        write_json(export_dir / 'checkpoints.json', {
            'best_sha256': sha256(run_dir / 'best.pt'),
            'latest_sha256': sha256(run_dir / 'latest.pt'),
            'pretrained_sha256': sha256(references[seed]['run'] / 'pretrain/best.pt')})
        archive = shutil.make_archive(str(ARCHIVE_ROOT / f'{name}_{RUN_MODE}'), 'zip',
                                      root_dir=EXPERIMENT_ROOT, base_dir=f'runs/{name}')
        shutil.make_archive(str(ARCHIVE_ROOT / f'adrec_diffusion_results_{RUN_MODE}'), 'zip', root_dir=RESULTS_DIR)
        print('Saved:', archive)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

## 8. Compare test accuracy and measure inference time

Re-evaluate all three checkpoint types using the same dataset, evaluation batch size, and seed convention. The original 32-step metrics are preserved alongside the newly measured results. Different GPU or software versions can change stochastic results slightly.

Timing measures model prediction, full-catalog scoring, and GPU top-20 selection. Inputs are already padded and on the device. Model loading, disk access, CPU-to-GPU transfer, and web-app overhead are excluded. CUDA synchronization surrounds each measurement; warm-up calls are excluded. Batch size 1 measures one-request latency; batch size 32 measures batch throughput. Do not run other GPU workloads during this cell.

All step counts use identical user histories and batch shapes. A fixed shuffled run order reduces ordering effects. Each execution writes a new measurement session so timings from different machines or sessions are not silently combined. Standard deviations across seeds are distinct from within-run p95 latency. Smoke measurements are pipeline checks only.

In [ ]:
limit = 9 if RUN_MODE == 'smoke' else None
comparison_data = {split: rows[:limit] for split, rows in data.items()} if limit else data
comparison_dataset = SequenceDataset(comparison_data, 'test', reference_cfg['max_len'])
eval_batch_size = 4 if RUN_MODE == 'smoke' else reference_cfg['evaluation_batch_size']
num_items = dataset_summary(data)['max_item_id']
torch.set_float32_matmul_precision('high')
SESSION_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SESSION_DIR = RESULTS_DIR / 'comparisons' / SESSION_ID
SESSION_DIR.mkdir(parents=True)
runtime = {'session_id': SESSION_ID, 'mode': RUN_MODE, 'device': DEVICE,
           'python': sys.version, 'torch': str(torch.__version__), 'cuda': torch.version.cuda,
           'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
           'float32_matmul_precision': torch.get_float32_matmul_precision(),
           'dataset_sha256': sha256(DATASET_PATH), 'source_manifest': manifest,
           'evaluation_batch_size': eval_batch_size, 'benchmark_seed': BENCHMARK_SEED,
           'batch_sizes': BENCHMARK_BATCH_SIZES, 'warmups': WARMUP_REPEATS,
           'repeats': TIMED_REPEATS, 'requested_timing_users': BENCHMARK_USERS,
           'timing_scope': 'Device-resident padded histories -> model -> full scores -> device top20; synchronized wall time; excludes loading, transfers, and app overhead.'}
write_json(SESSION_DIR / 'environment.json', runtime)
order = [(seed, steps) for seed in SEEDS for steps in [8, 16, 32]]
random.Random(BENCHMARK_SEED).shuffle(order)
write_json(SESSION_DIR / 'run_order.json', order)
accuracy_rows, latency_rows, raw_timings = [], [], []
for seed, steps in order:
    folder = references[seed]['run'] / 'adrec' if steps == 32 else variant_runs[(seed, steps)]
    model, saved_cfg = load_for_inference(folder / 'best.pt', DEVICE)
    expected_cfg = baseline_configs[seed] if steps == 32 else variant_config(baseline_configs[seed], steps, RUN_MODE)
    if saved_cfg != expected_cfg:
        raise ValueError(f'Checkpoint settings mismatch: {folder}')
    checkpoint_hash = sha256(folder / 'best.pt')
    seed_everything(seed)
    report = evaluate(comparison_dataset, lambda histories, masks: predict_scores(model, histories, masks),
                      num_items, batch_size=eval_batch_size, ks=reference_cfg['metric_ks'])
    accuracy_rows.append({'model': 'ADRec', 'mode': RUN_MODE, 'seed': seed,
                         'diffusion_steps': steps, **report['metrics']})
    saved_report = read_json(folder / 'results.json')
    write_json(SESSION_DIR / f'steps{steps}_seed{seed}_evaluation.json', {
        'seed': seed, 'diffusion_steps': steps, 'checkpoint_sha256': checkpoint_hash,
        'evaluation': report, 'original_saved_evaluation': saved_report.get('test'),
        'config': saved_cfg})
    for batch_size in BENCHMARK_BATCH_SIZES:
        batches = timing_batches(comparison_dataset, BENCHMARK_USERS, batch_size, DEVICE)
        batch_hash = hashlib.sha256(b''.join(x.cpu().numpy().tobytes() for x in batches)).hexdigest()
        timing = benchmark(model, batches, WARMUP_REPEATS, TIMED_REPEATS, BENCHMARK_SEED)
        raw_timings.append({'seed': seed, 'diffusion_steps': steps, 'checkpoint_sha256': checkpoint_hash,
                            'input_batches_sha256': batch_hash, **timing})
        latency_rows.append({'mode': RUN_MODE, 'seed': seed, 'diffusion_steps': steps,
                             **{k: v for k, v in timing.items() if k != 'samples_ms'}})
        del batches
    save_csv(SESSION_DIR / 'accuracy_runs.csv', accuracy_rows)
    save_csv(SESSION_DIR / 'latency_runs.csv', latency_rows)
    write_json(SESSION_DIR / 'raw_timings.json', raw_timings)
    shutil.make_archive(str(ARCHIVE_ROOT / f'adrec_diffusion_results_{RUN_MODE}'), 'zip', root_dir=RESULTS_DIR)
    print('Measured seed', seed, 'steps', steps, report['metrics'], flush=True)
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 9. Summarize the comparison

Means and sample standard deviations use matched seeds. Summary files include the actual seed list and run count; a subset is not labeled as five runs. Relative accuracy changes are percentages; absolute accuracy changes are percentage points. Speedup is the 32-step mean latency divided by the variant mean latency at the same batch size. Values above 1 indicate faster prediction.

Use validation for configuration selection. Report test results for the predefined experiment, including any accuracy loss. The observed trade-off, rather than a guaranteed improvement, is the outcome.

In [ ]:
expected_pairs = {(seed, steps) for seed in SEEDS for steps in [8, 16, 32]}
assert {(r['seed'], r['diffusion_steps']) for r in accuracy_rows} == expected_pairs
assert len(accuracy_rows) == len(expected_pairs)
assert {(r['seed'], r['diffusion_steps'], r['batch_size']) for r in latency_rows} == {
    (s, t, b) for s, t in expected_pairs for b in BENCHMARK_BATCH_SIZES}
accuracy_summary = summarize_rows(accuracy_rows, ['diffusion_steps'], METRICS)
latency_summary = summarize_rows(latency_rows, ['diffusion_steps', 'batch_size'],
                                ['mean_batch_ms', 'median_batch_ms', 'p95_batch_ms', 'users_per_second'])
baseline_accuracy = next(r for r in accuracy_summary if r['diffusion_steps'] == 32)
for row in accuracy_summary:
    for metric in METRICS:
        base = baseline_accuracy[metric + '_mean']
        row[metric + '_delta_pp_vs32'] = row[metric + '_mean'] - base
        row[metric + '_relative_change_pct_vs32'] = 100 * (row[metric + '_mean'] / base - 1) if base else None
baseline_latency = {r['batch_size']: r['mean_batch_ms_mean'] for r in latency_summary if r['diffusion_steps'] == 32}
for row in latency_summary:
    row['speedup_vs32'] = baseline_latency[row['batch_size']] / row['mean_batch_ms_mean']
save_csv(SESSION_DIR / 'accuracy_summary.csv', accuracy_summary)
save_csv(SESSION_DIR / 'latency_summary.csv', latency_summary)
write_json(SESSION_DIR / 'completion.json', {'completed': True, 'seeds': SEEDS, 'steps': [8, 16, 32], 'mode': RUN_MODE})
print('Accuracy (percent; mean +/- sample standard deviation):')
for row in accuracy_summary:
    print(row['diffusion_steps'], 'steps:', {m: (round(row[m + '_mean'], 4),
          None if row[m + '_std'] is None else round(row[m + '_std'], 4)) for m in METRICS})
print('\nInference time:')
for row in latency_summary:
    print(f"{row['diffusion_steps']} steps, batch {row['batch_size']}: "
          f"{row['mean_batch_ms_mean']:.3f} ms/batch; {row['speedup_vs32']:.2f}x vs 32 steps")
print('Complete session:', SESSION_DIR)

## 10. Download the experiment

The results ZIP contains per-variant metadata, all measurement sessions, timing samples, CSV summaries, and noise-schedule settings. The models ZIP contains the newly trained 8- and 16-step checkpoints, including resume state. Keep the original 32-step and pretraining archive separately.

Download both files from Kaggle's `/kaggle/working` output browser. Full results belong under `results/adrec_diffusion_steps/`; model runs belong under `checkpoints/adrec_diffusion_steps/`. Keep smoke outputs separate. For a new-session resume, extract the models ZIP and set `RESUME_ROOT` to its parent directory containing `runs/`.

`Save & Run All` starts a separate execution; it does not inherit this interactive session's output files. Save checkpoints before ending the session.

In [ ]:
results_zip = shutil.make_archive(str(ARCHIVE_ROOT / f'adrec_diffusion_results_{RUN_MODE}'), 'zip', root_dir=RESULTS_DIR)
models_zip = shutil.make_archive(str(ARCHIVE_ROOT / f'adrec_diffusion_models_{RUN_MODE}'), 'zip',
                                 root_dir=EXPERIMENT_ROOT, base_dir='runs')
try:
    from IPython.display import FileLink, display
except ImportError:  # Command-line smoke checks do not require IPython.
    FileLink, display = str, print
for archive in (results_zip, models_zip):
    print(archive)
    display(FileLink(archive))

## Compatibility with saved Baby runs

The original Baby training configuration and model math are retained. Source validation accepts only the exact old/new training-module hashes listed in `configs/legacy_source_compatibility.json`; all other source differences still fail validation. Accepted migrations are saved in `baseline_source_migrations.json` alongside the experiment results.
